## Imports

In [78]:
import time
import math
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import LinearSVR
from sklearn.metrics import (
    mean_absolute_error as mae,
    mean_squared_error as mse
)
import joblib

## Load Data

In [5]:
df = pd.read_csv("../data/train-test.csv")

print(f"Shape: {df.shape}")

df.head()

Shape: (48000, 14)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


## Inspect Data

In [6]:
print(f"Data metadata: \n{df.info()}")

<class 'pandas.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  str    
 1   pickup        48000 non-null  str    
 2   delivery      48000 non-null  str    
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  str    
 9   weight        47700 non-null  float64
 10  date          48000 non-null  str    
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), str(5)
memory usage: 5.1 MB
Data metadata: 
None


In [8]:
# Percentage of Missing Values
df.isna().sum() / len(df)

load_id         0.000000
pickup          0.000000
delivery        0.000000
pickup_lat      0.000000
pickup_lon      0.000000
delivery_lat    0.000000
delivery_lon    0.000000
distance        0.000000
equipment       0.000000
weight          0.006250
date            0.000000
market_index    0.007792
quote_signal    0.000000
posted_rate     0.000000
dtype: float64

In [9]:
df.duplicated().sum()

np.int64(0)

## Clean Data
- Missing Values -> Drop
    - As There are only 2 columns and each has small percentage of missings
- Duplicates -> Not Found
- Fix Data Types
    - Handle dates

In [ ]:
print(f"BEFORE: Shape = {df.shape}")
df = df.dropna()
print(f"AFTER: Shape = {df.shape}")

BEFORE: Shape = (48000, 14)
AFTER: Shape = (47327, 14)


In [11]:
print(f"BEFORE: Shape = {df.shape}")
df = df.drop_duplicates()
print(f"AFTER: Shape = {df.shape}")

BEFORE: Shape = (47327, 14)
AFTER: Shape = (47327, 14)


In [12]:
df.dtypes

load_id             str
pickup              str
delivery            str
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment           str
weight          float64
date                str
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object

In [13]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 47327 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   load_id       47327 non-null  str           
 1   pickup        47327 non-null  str           
 2   delivery      47327 non-null  str           
 3   pickup_lat    47327 non-null  float64       
 4   pickup_lon    47327 non-null  float64       
 5   delivery_lat  47327 non-null  float64       
 6   delivery_lon  47327 non-null  float64       
 7   distance      47327 non-null  float64       
 8   equipment     47327 non-null  str           
 9   weight        47327 non-null  float64       
 10  date          47327 non-null  datetime64[us]
 11  market_index  47327 non-null  float64       
 12  quote_signal  47327 non-null  float64       
 13  posted_rate   47327 non-null  float64       
dtypes: datetime64[us](1), float64(9), str(4)
memory usage: 5.4 MB


## Split Data
- Training -> Modeling
- Testing ->  Evaluation

In [26]:
df['pickup'].value_counts()

pickup
Oklahoma City    1226
Lexington        1193
Bakersfield      1174
Fort Wayne       1155
Hartford         1132
                 ... 
Washington        334
Detroit           331
St. Louis         315
Las Vegas         276
Dallas            272
Name: count, Length: 64, dtype: int64

In [27]:
df['delivery'].value_counts()

delivery
Lexington        1183
Fort Wayne       1154
Baton Rouge      1153
Bakersfield      1137
Oklahoma City    1123
                 ... 
St. Louis         341
Birmingham        317
Dallas            306
Detroit           304
Las Vegas         289
Name: count, Length: 64, dtype: int64

In [28]:
df['equipment'].value_counts()

equipment
Dry Van    26825
Reefer     11876
Flatbed     8626
Name: count, dtype: int64

In [31]:
le = LabelEncoder()
df['equipment'] = le.fit_transform(df['equipment'])

print("Category Mapping:", le.classes_)

Category Mapping: ['Dry Van' 'Flatbed' 'Reefer']


In [38]:
df['date'].unique()

<DatetimeArray>
['2025-01-01 00:00:00', '2025-01-02 00:00:00', '2025-01-03 00:00:00',
 '2025-01-04 00:00:00', '2025-01-05 00:00:00', '2025-01-06 00:00:00',
 '2025-01-07 00:00:00', '2025-01-08 00:00:00', '2025-01-09 00:00:00',
 '2025-01-10 00:00:00',
 ...
 '2025-10-22 00:00:00', '2025-10-23 00:00:00', '2025-10-24 00:00:00',
 '2025-10-25 00:00:00', '2025-10-26 00:00:00', '2025-10-27 00:00:00',
 '2025-10-28 00:00:00', '2025-10-29 00:00:00', '2025-10-30 00:00:00',
 '2025-10-31 00:00:00']
Length: 304, dtype: datetime64[us]

In [39]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

In [42]:
df['day'] = df['date'].dt.day

In [40]:
df['year'].unique()

array([2025], dtype=int32)

In [41]:
df['month'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10], dtype=int32)

In [43]:
df['day'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31],
      dtype=int32)

In [44]:
df.info()

<class 'pandas.DataFrame'>
Index: 47327 entries, 0 to 47999
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   load_id       47327 non-null  str           
 1   pickup        47327 non-null  str           
 2   delivery      47327 non-null  str           
 3   pickup_lat    47327 non-null  float64       
 4   pickup_lon    47327 non-null  float64       
 5   delivery_lat  47327 non-null  float64       
 6   delivery_lon  47327 non-null  float64       
 7   distance      47327 non-null  float64       
 8   equipment     47327 non-null  int64         
 9   weight        47327 non-null  float64       
 10  date          47327 non-null  datetime64[us]
 11  market_index  47327 non-null  float64       
 12  quote_signal  47327 non-null  float64       
 13  posted_rate   47327 non-null  float64       
 14  year          47327 non-null  int32         
 15  month         47327 non-null  int32         
 16  da

In [45]:
df.columns

Index(['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon',
       'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight',
       'date', 'market_index', 'quote_signal', 'posted_rate', 'year', 'month',
       'day'],
      dtype='str')

In [46]:
features = [
    'pickup_lat', 'pickup_lon',
    'delivery_lat', 'delivery_lon',
    'distance', 'weight',
    'market_index', 'quote_signal',
    'equipment', 'month'
]

In [47]:
# Define Features & Target
X = df[features]
y = df['posted_rate'].values

print(f"Number of Features = {X.shape[1]}")

Number of Features = 10


In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

print(f"Training Data Shape = {X_train.shape}")
print(f"Testing Data Shape = {X_test.shape}")

Training Data Shape = (35495, 10)
Testing Data Shape = (11832, 10)


In [49]:
print(f"Training Data Size = {(X_train.shape[0] / len(df))*100:.2f}%")
print(f"Testing Data Size = {(X_test.shape[0] / len(df))*100:.2f}%")

Training Data Size = 75.00%
Testing Data Size = 25.00%


In [50]:
X_train.head()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,equipment,month
31445,43.96975,-71.05842,38.87577,-77.05086,566.2,40223.0,1.32704,1.96589,0,7
12812,31.98691,-84.48897,33.45894,-94.78556,711.1,-18229.0,0.98671,2.02344,0,3
24551,35.19670,-121.69849,33.45894,-94.78556,1780.0,30772.0,1.33005,1.96695,0,6
1111,39.22317,-72.96710,43.96975,-71.05842,423.1,44641.0,0.97762,2.41583,2,1
15286,39.27463,-69.50000,35.19670,-121.69849,3284.2,36794.0,0.98126,2.37687,0,4


## Machine Learning Modeling

In [51]:
model_portfolio = {
    'Linear Regression': LinearRegression(
        fit_intercept=True,
        n_jobs=-1
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        n_jobs=-1,
        random_state=42
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=2,
        subsample=0.8,
        random_state=42
    ),
    'Support Vector': LinearSVR(
        C=1.0,
        epsilon=0.1,
        max_iter=5000,
        dual=True,
        random_state=42
    )
}

In [54]:
print("Start Models Training.....")

results = {}
for model_name, model in model_portfolio.items():
    print(f"    {model_name} Training....")
    t0 = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - t0
    print(f"    Training Time = {training_time:.2f} seconds")
    y_pred = model.predict(X_test)
    y_pred_train = model.predict(X_train)
    
    results[model_name] = {
        'training_time': training_time,
        'train_r2': model.score(X_train, y_train),
        'test_r2': model.score(X_test, y_test),
        'train_mae': mae(y_train, y_pred_train),
        'test_mae': mae(y_test, y_pred),
        'train_mse': mse(y_train, y_pred_train),
        'test_mse': mse(y_test, y_pred)
    }
    print("Model Trained Successfully")
    print("_"*50)

Start Models Training.....
    Linear Regression Training....
    Training Time = 0.07 seconds
Model Trained Successfully
__________________________________________________
    Random Forest Training....
    Training Time = 1.51 seconds
Model Trained Successfully
__________________________________________________
    Gradient Boosting Training....
    Training Time = 11.19 seconds
Model Trained Successfully
__________________________________________________
    Support Vector Training....
    Training Time = 28.61 seconds
Model Trained Successfully
__________________________________________________


e:\DataScience Bootcamp\Different Platforms\Flyrank.AI\assessment\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [60]:
print("Models Comparison")

results_df = pd.DataFrame(results).T
results_df = results_df.round(4).sort_values('test_r2', ascending=False)

results_df

Models Comparison


,training_time,train_r2,test_r2,train_mae,test_mae,train_mse,test_mse
Gradient Boosting,11.1881,0.8703,0.8283,108.2568,118.7390,286139.1303,383217.8952
Random Forest,1.5072,0.9067,0.8269,105.7042,133.9888,205934.8727,386202.9159
Linear Regression,0.0722,0.8385,0.8258,140.2731,143.4518,356273.8413,388758.0977
Support Vector,28.6145,0.8240,0.8110,191.3606,194.5211,388255.3798,421726.5232


## Hyperparameter Tuning

In [62]:
param_grids = {
    'Linear Regression': {
        'fit_intercept': [True, False]
    },

    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [8, 12, 16, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2']
    },

    'Gradient Boosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5],
        'subsample': [0.7, 0.8, 1.0]
    },
}

In [63]:
best_models = {}

for name, model in model_portfolio.items():
    if name == 'Support Vector':
        continue
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grids[name],
        n_iter=20,
        scoring='neg_mean_squared_error',
        cv=5,
        n_jobs=-1,
        random_state=42,
        verbose=1
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    print(f"{name} → Best Score: {-search.best_score_:.4f}")
    print(f"Best Params: {search.best_params_}\n")
    print("_"*50)

e:\DataScience Bootcamp\Different Platforms\Flyrank.AI\assessment\.venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 2 is smaller than n_iter=20. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 2 candidates, totalling 10 fits
Linear Regression → Best Score: 356499.0012
Best Params: {'fit_intercept': False}

__________________________________________________
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Random Forest → Best Score: 350726.3825
Best Params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': None}

__________________________________________________
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Gradient Boosting → Best Score: 349605.4566
Best Params: {'subsample': 0.7, 'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 3, 'learning_rate': 0.05}

__________________________________________________


In [75]:
best_models

{'Linear Regression': LinearRegression(fit_intercept=False, n_jobs=-1),
 'Random Forest': RandomForestRegressor(max_features='sqrt', min_samples_leaf=4, n_estimators=300,
                       n_jobs=-1, random_state=42),
 'Gradient Boosting': GradientBoostingRegressor(learning_rate=0.05, min_samples_leaf=2,
                           min_samples_split=5, n_estimators=200,
                           random_state=42, subsample=0.7)}

## Save the Best Model

In [76]:
# Save the model
best_model = best_models['Gradient Boosting']
joblib.dump(best_model, 'model.joblib')

['model.joblib']

In [77]:
# Load the model
loaded_model = joblib.load('model.joblib')

# Use the loaded model
print(loaded_model.score(X_test, y_test))

0.8296720812090299
